In [9]:
import os
import sys
from dotenv import load_dotenv

load_dotenv()

from typing import Literal
from pydantic import BaseModel
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, AIMessage
from langgraph.graph import MessagesState, StateGraph, START
from langgraph.types import Command

# Load config
sys.path.insert(0, os.path.join(os.getcwd(), "utils"))
from utils.config_loader import load_config

config = load_config()
llm_config = config["llm"]

def build_llm(agent_key: str) -> ChatGroq:
    """Build a ChatGroq LLM instance from config for a given agent."""
    cfg = llm_config[agent_key]
    return ChatGroq(
        model=cfg["model_name"],
        temperature=cfg.get("temperature", 0),
        api_key=os.getenv("GROQ_API_KEY"),
    )


In [10]:
# Define the Supervisor's Routing Logic
llm = build_llm("agent1")

# This Pydantic model helps the supervisor choose the next agent.
class Router(BaseModel):
    """Decide the next agent to route to."""
    next_agent: Literal["Weather", "Flights", "__end__"]

# Bind the router to the model to get structured output.
supervisor_model = llm.with_structured_output(Router)

def supervisor(state: MessagesState) -> Command:
    """
    The central supervisor that routes to the correct agent or ends the conversation.
    """
    print("--- 🧑‍💼 SUPERVISOR ---")
    
    # The prompt tells the supervisor how to route the user's message.
    prompt = f"""You are a supervisor routing tasks to a specialist agent. 
    Based on the user's request, choose the appropriate agent.
    
    Available agents:
    - Weather: For questions about weather forecasts.
    - Flights: For questions about flight information.
    
    If the user is saying thank you or the conversation is over, choose '__end__'.
    
    User message: "{state['messages'][-1].content}"
    """
    if isinstance(state['messages'][-1], HumanMessage):
      # The supervisor model makes the routing decision.
      response = supervisor_model.invoke(prompt)
      print(f"Supervisor routing to: {response.next_agent}")
      return Command(goto=response.next_agent)
    else:
      return Command(goto='__end__')

# Define the Specialist Agents

def weather_agent(state: MessagesState) -> Command:
    """A specialist agent for handling weather-related queries."""
    print("--- ☀️ WEATHER AGENT ---")
    
    prompt = f"""You are a weather forecaster. Provide a concise mock weather forecast 
    for the location mentioned in the user's message.
    
    User message: "{state['messages'][-1].content}"
    """
    response = llm.invoke(prompt)
    print(f"Response: {response.content}")
    # Return to the supervisor after the agent has run.
    return Command(
        goto="supervisor",
        update={"messages": [response]},
    )

def flights_agent(state: MessagesState) -> Command:
    """A specialist agent for handling flight-related queries."""
    print("--- ✈️ FLIGHTS AGENT ---")
    
    prompt = f"""You are a flight information assistant. Provide some mock flight
    details for the destination in the user's message. Respond with short concise information.
    
    User message: "{state['messages'][-1].content}"
    """

    response = llm.invoke(prompt)
    print(f"Response: {response.content}")
    # Return to the supervisor after the agent has run.
    return Command(
        goto="supervisor",
        update={"messages": [response]},
    )

# Build the Graph

builder = StateGraph(MessagesState)

# Add the supervisor and agents as nodes in the graph.
builder.add_node("supervisor", supervisor)
builder.add_node("Weather", weather_agent)
builder.add_node("Flights", flights_agent)

# The START node directs the flow to the supervisor first.
builder.add_edge(START, "supervisor")

# The graph is now complete. The `Command` objects will handle the dynamic routing.
graph = builder.compile()